# AI Financial Risk Intelligence Platform

## Notebook 01: Data Understanding

### Business Problem

A financial institution receives loan applications from customers. Granting loans to applicants who are likely to default can lead to significant financial losses.

Our goal is to build a machine learning system that predicts whether a loan applicant is **high risk (bad credit)** or **low risk (good credit)** before approving the loan.

### Objective

Predict the probability that a customer belongs to the high-risk class and provide an explanation for the prediction.

### Success Metric

* Primary metric: ROC-AUC
* Secondary metrics: Precision, Recall, F1-score
* Business goal: Minimize high-risk loans approved by the bank

### ML Problem Type

* Supervised Learning
* Binary Classification
* Baseline model: Logistic Regression


In [3]:
from pathlib import Path
import pandas as pd

# Project root
PROJECT_ROOT = Path.cwd().resolve().parent

# Raw data path
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "german.data"

column_names = [
    "status", "duration", "credit_history", "purpose", "credit_amount",
    "savings", "employment_duration", "installment_rate", "personal_status_sex",
    "other_debtors", "present_residence", "property", "age",
    "other_installment_plans", "housing", "existing_credits", "job",
    "people_liable", "telephone", "foreign_worker", "target"
]

df = pd.read_csv(
    DATA_PATH,
    sep=r"\s+",
    header=None,
    names=column_names
)

print("Dataset loaded successfully")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully
Shape: (1000, 21)


,status,duration,credit_history,purpose,credit_amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,existing_credits,job,people_liable,telephone,foreign_worker,target
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


In [4]:
# ============================================================
# Basic Inspection
# ============================================================

print("Rows and Columns:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nFirst 5 Rows:")
display(df.head())

Rows and Columns:
(1000, 21)

Column Names:
['status', 'duration', 'credit_history', 'purpose', 'credit_amount', 'savings', 'employment_duration', 'installment_rate', 'personal_status_sex', 'other_debtors', 'present_residence', 'property', 'age', 'other_installment_plans', 'housing', 'existing_credits', 'job', 'people_liable', 'telephone', 'foreign_worker', 'target']

Data Types:
status                     object
duration                    int64
credit_history             object
purpose                    object
credit_amount               int64
savings                    object
employment_duration        object
installment_rate            int64
personal_status_sex        object
other_debtors              object
present_residence           int64
property                   object
age                         int64
other_installment_plans    object
housing                    object
existing_credits            int64
job                        object
people_liable               int64
telep

,status,duration,credit_history,purpose,credit_amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,existing_credits,job,people_liable,telephone,foreign_worker,target
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


In [5]:
# ============================================================
# Target Variable Distribution
# ============================================================

target_counts = df["target"].value_counts().sort_index()

print(target_counts)

print("\nTarget Distribution (%):")
print((target_counts / len(df) * 100).round(2))

target
1    700
2    300
Name: count, dtype: int64

Target Distribution (%):
target
1    70.0
2    30.0
Name: count, dtype: float64


In [6]:
# ============================================================
# Convert Target to Binary
# 0 = Good Credit
# 1 = Bad Credit
# ============================================================

df["target"] = df["target"].map({1: 0, 2: 1})

print(df["target"].value_counts())

print("\nBad Credit Rate:", df["target"].mean().round(3))

target
0    700
1    300
Name: count, dtype: int64

Bad Credit Rate: 0.3


In [7]:
# ============================================================
# Class Balance Check
# ============================================================

class_percent = (
    df["target"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(class_percent)

if class_percent.min() < 40:
    print("\nDataset is imbalanced.")
else:
    print("\nDataset is relatively balanced.")

target
0    70.0
1    30.0
Name: proportion, dtype: float64

Dataset is imbalanced.


In [8]:
# ============================================================
# Missing Values and Feature Types
# ============================================================

print("Missing Values:")
print(df.isnull().sum())

categorical_cols = df.select_dtypes(include="object").columns.tolist()
numerical_cols = df.select_dtypes(exclude="object").drop(columns="target").columns.tolist()

print("\nCategorical Columns:")
print(categorical_cols)

print("\nNumerical Columns:")
print(numerical_cols)

print("\nNumber of Categorical Features:", len(categorical_cols))
print("Number of Numerical Features:", len(numerical_cols))

Missing Values:
status                     0
duration                   0
credit_history             0
purpose                    0
credit_amount              0
savings                    0
employment_duration        0
installment_rate           0
personal_status_sex        0
other_debtors              0
present_residence          0
property                   0
age                        0
other_installment_plans    0
housing                    0
existing_credits           0
job                        0
people_liable              0
telephone                  0
foreign_worker             0
target                     0
dtype: int64

Categorical Columns:
['status', 'credit_history', 'purpose', 'savings', 'employment_duration', 'personal_status_sex', 'other_debtors', 'property', 'other_installment_plans', 'housing', 'job', 'telephone', 'foreign_worker']

Numerical Columns:
['duration', 'credit_amount', 'installment_rate', 'present_residence', 'age', 'existing_credits', 'people_liable']

Nu

In [9]:
# ============================================================
# Separate Features and Target
# ============================================================

X = df.drop(columns="target")
y = df["target"]

print("Feature Shape:", X.shape)
print("Target Shape :", y.shape)

X.head()

Feature Shape: (1000, 20)
Target Shape : (1000,)


,status,duration,credit_history,purpose,credit_amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,present_residence,property,age,other_installment_plans,housing,existing_credits,job,people_liable,telephone,foreign_worker
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,4,A121,67,A143,A152,2,A173,1,A192,A201
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,2,A121,22,A143,A152,1,A173,1,A191,A201
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,3,A121,49,A143,A152,1,A172,2,A191,A201
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,4,A122,45,A143,A153,1,A173,2,A191,A201
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,4,A124,53,A143,A153,2,A173,2,A191,A201


In [10]:
# ============================================================
# Preprocessing Imports
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

print("Preprocessing libraries imported successfully")

Preprocessing libraries imported successfully


In [11]:
# ============================================================
# ColumnTransformer
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_cols
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ]
)

print(preprocessor)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['duration', 'credit_amount',
                                  'installment_rate', 'present_residence',
                                  'age', 'existing_credits', 'people_liable']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['status', 'credit_history', 'purpose',
                                  'savings', 'employment_duration',
                                  'personal_status_sex', 'other_debtors',
                                  'property', 'other_installment_plans',
                                  'housing', 'job', 'telephone',
                                  'foreign_worker'])])


## Key Findings

* The dataset contains **1000 loan applicants** and **20 input features**.
* The target variable has been converted to a binary format:

  * **0 = Good Credit**
  * **1 = Bad Credit**
* There are **13 categorical features** and **7 numerical features**.
* The dataset contains **no missing values**.
* The target distribution is **imbalanced**, making **ROC-AUC, Precision, Recall, and F1-score** more appropriate than accuracy alone.
* The presence of many categorical variables indicates that a **OneHotEncoder + ColumnTransformer + Pipeline** architecture will be required for the Logistic Regression model.

### Next Notebook

`02_eda.ipynb` will perform exploratory data analysis to understand feature distributions, relationships, and business insights before any preprocessing or modeling is performed.
